# M:M Quality Analysis

Loads all saved experiment results and produces the analysis tables and
visualisations needed for the thesis.  **Does not call schema_match() or
any LLM** — it only reads from `results/` files.

In [ ]:
# ── Cell 1: environment setup ──────────────────��─────────────────────────────
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

_cwd = Path(os.getcwd()).resolve()
for _candidate in [_cwd, _cwd.parent, _cwd / "thesis-extension"]:
    if (_candidate / "pipeline.py").exists():
        _thesis_root = _candidate
        break
else:
    raise RuntimeError(f"Cannot locate thesis-extension root from CWD={_cwd}.")

_notebooks_dir = _thesis_root / "notebooks"
_project_root  = _thesis_root.parent
_results_dir   = _thesis_root / "results"
_gt_path       = _project_root / "updated_ground_truth.csv"

os.environ.setdefault("RESULTS_DIR",  str(_results_dir))
os.environ.setdefault("TEMPLATE_DIR", str(_thesis_root / "templates"))
load_dotenv(_thesis_root / ".env", override=False)

for _d in [str(_thesis_root), str(_notebooks_dir)]:
    if _d not in sys.path:
        sys.path.insert(0, _d)

print("thesis-extension root :", _thesis_root)
print("Results dir           :", _results_dir)
print("Ground truth path     :", _gt_path)

In [ ]:
# ── Cell 2: imports ──────────────────────��───────────────────────────────────
import json
import warnings
from datetime import date

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from ground_truth import load_updated_ground_truth
from models import Result, Vote

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ── Cell 3: data loading ───────────────────���─────────────────────────────────
def _load_csv(name: str) -> pd.DataFrame | None:
    p = _results_dir / f"{name}.csv"
    if not p.exists():
        print(f"  WARNING: {p.name} not found — section skipped")
        return None
    df = pd.read_csv(p)
    print(f"  {p.name}: {len(df)} rows")
    return df

def _load_json(name: str) -> dict | None:
    p = _results_dir / f"{name}.json"
    if not p.exists():
        print(f"  WARNING: {p.name} not found")
        return None
    data = json.loads(p.read_text(encoding="utf-8"))
    print(f"  {p.name}: loaded")
    return data

def _load_jsonl(name: str) -> pd.DataFrame | None:
    p = _results_dir / f"{name}.jsonl"
    if not p.exists():
        print(f"  WARNING: {p.name} not found")
        return None
    lines = [l for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not lines:
        print(f"  {p.name}: empty")
        return pd.DataFrame()
    df = pd.DataFrame([json.loads(l) for l in lines])
    print(f"  {p.name}: {len(df)} records")
    return df

print("Loading results files:")
df_baseline     = _load_csv("claude_baseline_results")
df_k2           = _load_csv("m2m_k2_results")
df_k3           = _load_csv("m2m_k3_results")
df_relatedness  = _load_csv("relatedness_results")
rel_metrics     = _load_json("relatedness_metrics")
df_cost         = _load_jsonl("cost_log")

In [ ]:
# ── Section 1: per-relation-pair F1 heatmap ───────────────────��──────────────
if df_baseline is None:
    print("Section 1 skipped — claude_baseline_results.csv missing.")
else:
    _metric_cols = [
        "precision_1to1", "recall_1to1", "f1_1to1",
        "precision_group", "recall_group", "f1_group",
    ]
    # Ensure all metric columns exist (fill with 0 if absent)
    for c in _metric_cols:
        if c not in df_baseline.columns:
            df_baseline[c] = 0.0

    _heat = df_baseline.set_index("pair")[_metric_cols]

    print("\nPer-relation-pair metrics:")
    print(_heat.to_string())
    print(f"\nMean F1 (1:1)  : {_heat['f1_1to1'].mean():.3f}")
    print(f"Mean F1 (group): {_heat['f1_group'].mean():.3f}")

    fig_heat = px.imshow(
        _heat,
        color_continuous_scale=[[0, "red"], [0.5, "yellow"], [1, "green"]],
        zmin=0, zmax=1,
        aspect="auto",
        title="Per-relation-pair metrics (Claude baseline)",
        labels={"color": "score"},
        text_auto=".2f",
    )
    fig_heat.update_layout(xaxis_tickangle=-30)
    fig_heat.write_html(str(_results_dir / "per_pair_heatmap.html"))
    print(f"\nSaved: {_results_dir / 'per_pair_heatmap.html'}")
    fig_heat.show()

In [ ]:
# ── Section 2: group match discovery ────────────────────────────���────────────
# Load ground truth group entries
if not _gt_path.exists():
    print("Section 2 skipped — updated_ground_truth.csv not found.")
else:
    _, _gt_groups = load_updated_ground_truth(str(_gt_path))

    # Build a lookup: frozenset(src_attr_names) x frozenset(tgt_attr_names) -> True
    def _attr_name(path: str) -> str:
        return path.rsplit(".", 1)[-1]

    _gt_group_keys = {
        (
            frozenset(_attr_name(s) for s in e.sources),
            frozenset(_attr_name(t) for t in e.targets),
        )
        for e in _gt_groups
    }

    # Scan all cached Result JSON files for group_pairs
    _cached_results_dir = _results_dir / "results"
    _found_groups: list[dict] = []

    if _cached_results_dir.exists():
        for _f in sorted(_cached_results_dir.glob("*.json")):
            try:
                _r = Result.from_json(_f.read_text(encoding="utf-8"))
            except Exception:
                continue
            if not _r.group_pairs:
                continue
            _src_n = _r.parameters.source_relation.name
            _tgt_n = _r.parameters.target_relation.name
            _k     = _r.parameters.max_group_size
            for agp, rgp in _r.group_pairs.items():
                _src_attrs = frozenset(a.name for a in agp.sources)
                _tgt_attrs = frozenset(a.name for a in agp.targets)
                _yes = sum(1 for d in rgp.votes if d.vote == Vote.YES)
                _score = _yes / len(rgp.votes) if rgp.votes else 0.0
                _is_tp = (_src_attrs, _tgt_attrs) in _gt_group_keys
                _found_groups.append({
                    "relation_pair": f"{_src_n}->{_tgt_n}",
                    "k": _k,
                    "sources": sorted(_src_attrs),
                    "targets": sorted(_tgt_attrs),
                    "score": round(_score, 3),
                    "votes": len(rgp.votes),
                    "label": "TP" if _is_tp else "FP",
                })
    else:
        print("  No cached results directory found.")

    # Determine false negatives: GT groups not found in any result
    _predicted_keys = {
        (frozenset(g["sources"]), frozenset(g["targets"]))
        for g in _found_groups
    }
    _fn_groups = [
        e for e in _gt_groups
        if (
            frozenset(_attr_name(s) for s in e.sources),
            frozenset(_attr_name(t) for t in e.targets),
        ) not in _predicted_keys
    ]

    print("\n=== Section 2: Group match discovery ===")
    if _found_groups:
        _df_groups = pd.DataFrame(_found_groups)
        for _, row in _df_groups.iterrows():
            print(
                f"  [{row['label']}] {row['relation_pair']} (K={row['k']}) "
                f"{row['sources']} -> {row['targets']} "
                f"score={row['score']:.3f} votes={row['votes']}"
            )
    else:
        print("  No group pairs discovered (mock mode or pipeline not yet run).")

    if _fn_groups:
        print("\nFalse negatives (GT group not discovered):")
        for e in _fn_groups:
            _src_names = [_attr_name(s) for s in e.sources]
            _tgt_names = [_attr_name(t) for t in e.targets]
            print(f"  [FN] {_src_names} -> {_tgt_names} | {e.relationship}")
    else:
        print("\nNo false negatives — all GT groups were discovered.")

In [ ]:
# ── Section 3: K=2 vs K=3 comparison ─────────────────────────────────────────
if df_k2 is None or df_k3 is None:
    print("Section 3 skipped — m2m_k2_results.csv or m2m_k3_results.csv missing.")
else:
    _cmp = df_k2[["pair", "f1_1to1", "f1_group", "group_pairs_found"]].merge(
        df_k3[["pair", "f1_1to1", "f1_group", "group_pairs_found"]],
        on="pair", suffixes=("_k2", "_k3"),
    )

    print("=== Section 3: K=2 vs K=3 comparison ===")
    print(_cmp.to_string(index=False))

    # Observations
    print()
    for _, row in _cmp.iterrows():
        delta = round(row["f1_group_k3"] - row["f1_group_k2"], 3)
        if delta > 0:
            obs = f"improved by {delta:.3f}"
        elif delta < 0:
            obs = f"degraded by {abs(delta):.3f}"
        else:
            obs = "unchanged"
        print(f"  {row['pair']:40s} f1_group {obs}")

    # Bar chart: f1_group at K=2 vs K=3
    _bar_data = pd.concat([
        df_k2[["pair", "f1_group"]].assign(k="K=2"),
        df_k3[["pair", "f1_group"]].assign(k="K=3"),
    ])

    fig_bar = px.bar(
        _bar_data,
        x="pair", y="f1_group", color="k",
        barmode="group",
        title="F1 (group) at K=2 vs K=3 per relation pair",
        labels={"f1_group": "F1 (group)", "pair": "Relation pair", "k": "K"},
        color_discrete_map={"K=2": "steelblue", "K=3": "darkorange"},
        range_y=[0, 1.05],
    )
    fig_bar.update_layout(xaxis_tickangle=-40)
    fig_bar.write_html(str(_results_dir / "k2_vs_k3_comparison.html"))
    print(f"\nSaved: {_results_dir / 'k2_vs_k3_comparison.html'}")
    fig_bar.show()

In [ ]:
# ── Section 4: cost analysis ─────────────────────────────────────────────────
print("=== Section 4: Cost analysis ===")
if df_cost is None or df_cost.empty:
    print("  cost_log.jsonl is empty or missing — real API calls have not been made yet.")
    _total_cost_usd = 0.0
else:
    # Check if all costs are zero (mock mode)
    _real_calls = df_cost[df_cost.get("cost_usd", 0) > 0] if "cost_usd" in df_cost.columns else pd.DataFrame()
    if len(_real_calls) == 0:
        print("  All cost_log entries have cost_usd=0 — this is mock-mode data.")
        print("  Re-run experiment notebooks with QUERY_LLM=True to record real costs.")
        _total_cost_usd = 0.0
    else:
        # Per-provider summary
        _by_provider = (
            df_cost.groupby("provider")
            .agg(
                calls=("prompt_digest", "count"),
                input_tokens=("input_tokens", "sum"),
                output_tokens=("output_tokens", "sum"),
                cost_usd=("cost_usd", "sum"),
                avg_latency_ms=("latency_ms", "mean"),
            )
            .round(4)
        )
        print("\nPer-provider:")
        print(_by_provider.to_string())
        _total_cost_usd = float(df_cost["cost_usd"].sum())
        print(f"\nTotal API calls     : {len(df_cost)}")
        print(f"Total cost (USD)    : ${_total_cost_usd:.4f}")
        print(f"Avg latency (ms)    : {df_cost['latency_ms'].mean():.1f}")

        # Cost per pair requires joining with Parameters — approximate via equal split
        _pairs_run = len(df_cost) // max(1, 9)  # rough: calls / 9 pairs
        print(f"Approx calls/pair   : {_pairs_run}")

In [ ]:
# ── Section 5: cross-LLM comparison ─────────────────────────���────────────────
print("=== Section 5: Cross-LLM comparison ===")

# Look for a GPT-4 baseline results file
_gpt4_csv = _results_dir / "gpt4_baseline_results.csv"
if not _gpt4_csv.exists():
    print(
        "  GPT-4 baseline results not yet available.\n"
        "  To generate them: run Marcel's notebooks from actual-repo/ "
        "  (generate_gpt4_results.ipynb) and copy the output to "
        f"  {_gpt4_csv.name} in results/.\n"
        "  Once present, re-run this cell to see the side-by-side comparison."
    )
elif df_baseline is None:
    print("  Claude baseline results missing — cannot compare.")
else:
    _df_gpt4 = pd.read_csv(_gpt4_csv)
    _cmp_llm = df_baseline[["pair", "f1_1to1", "f1_group"]].merge(
        _df_gpt4[["pair", "f1_1to1", "f1_group"]],
        on="pair", suffixes=("_claude", "_gpt4"),
    )
    print("Claude vs GPT-4 F1 comparison:")
    print(_cmp_llm.to_string(index=False))
    print(f"\nClaude mean F1 (1:1) : {df_baseline['f1_1to1'].mean():.3f}")
    print(f"GPT-4  mean F1 (1:1) : {_df_gpt4['f1_1to1'].mean():.3f}")

In [ ]:
# ── Section 6: thesis results summary ───────────────────���────────────────────
print("=== Section 6: Thesis results summary ===")

def _mean_or_null(df: pd.DataFrame | None, col: str):
    if df is None or col not in df.columns:
        return None
    v = float(df[col].mean())
    return round(v, 4)

summary = {
    "claude_baseline_f1_1to1":   _mean_or_null(df_baseline, "f1_1to1"),
    "m2m_k2_f1_group":           _mean_or_null(df_k2, "f1_group"),
    "m2m_k3_f1_group":           _mean_or_null(df_k3, "f1_group"),
    "relatedness_f1":            (
        round(rel_metrics["f1"], 4) if rel_metrics else None
    ),
    "cost_total_usd":            (
        round(_total_cost_usd, 4)
        if df_cost is not None and not df_cost.empty and "cost_usd" in df_cost.columns
        else None
    ),
    "relation_pairs_evaluated":  (
        int(len(df_baseline)) if df_baseline is not None else None
    ),
    "date_generated":            str(date.today()),
}

_summary_path = _results_dir / "thesis_results_summary.json"
_summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"\nSaved: {_summary_path}")